### Notebook Purpose: Results Consolidation
- Consolidate per-model metric tables into a single comparison dataframe
- Pull baseline/model rows from CSV exports in `results/metrics/`
- Save the consolidated CSV and display quick sanity plots for reporting
- Provide a lightweight summary artifact for presentations or sign-off

### Note
- Shared inputs/outputs and execution conventions are documented in the project README.


In [1]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# Minimal environment setup to keep the notebook focused on the core experiments.
# If you run this elsewhere, update PROJECT_DIR accordingly.
PROJECT_DIR = Path("/content/drive/MyDrive/0.Portfolio/electricity_price_forecasting").resolve()


Mounted at /content/drive


In [2]:
project_str = str(PROJECT_DIR)
if project_str not in sys.path:
    sys.path.insert(0, project_str)
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


DATA_DIR         = PROJECT_DIR / "data"
RESULTS_DIR      = PROJECT_DIR / "results"
METRICS_DIR      = RESULTS_DIR / "metrics"
MODELS_DIR       = RESULTS_DIR / "models"
WINDOW_INDEX_DIR = RESULTS_DIR / "window_index"
LOGS_DIR         = RESULTS_DIR / "logs"
CACHE_DIR        = RESULTS_DIR / "cache"
PLOTS_DIR        = RESULTS_DIR / "plots"

for p in [
    RESULTS_DIR,
    PLOTS_DIR,
    METRICS_DIR,
    MODELS_DIR,
    WINDOW_INDEX_DIR,
    LOGS_DIR,
    CACHE_DIR,
    PLOTS_DIR / "models",
    PLOTS_DIR / "eval/curves",
    PLOTS_DIR / "eval/tables",
    PLOTS_DIR / "eda",
    PLOTS_DIR / "feature_engineering",
]:
    p.mkdir(parents=True, exist_ok=True)


# 06_Results_Comparison

This notebook builds the final cross-model metrics table and comparison plots.


In [3]:
# === Build consolidated metrics table ===

# from the exported model-vs-baseline CSVs.
# - Model rows: from each file in IN_FILES
# - Baseline row: ONLY from metrics_table_dl_gated_transformer_hurdle_vs_baseline.csv
# - Output columns: model_name + VALUE_COLS (+ optional source_file)

import pandas as pd
from pathlib import Path

METRICS_DIR = RESULTS_DIR / "metrics"

IN_FILES = [
    "metrics_table_dl_hurdle_tcn_vs_baseline.csv",
    "metrics_table_ml_hurdle_lgbm_vs_baseline.csv",
    "metrics_table_dl_gated_transformer_hurdle_vs_baseline.csv",
    "metrics_table_dl_hurdle_lstm_vs_baseline.csv",
]

BASELINE_SOURCE_FILE = "metrics_table_dl_gated_transformer_hurdle_vs_baseline.csv"

VALUE_COLS = [
    "Active High", "Active Low", "Active Close", "Active Volume", "Active Total",
    "Full High", "Full Low", "Full Close", "Full Volume", "Full Total",
]

# Optional: normalize names to stable ids for plotting/merging
MODEL_NAME_BY_FILE = {
    "metrics_table_dl_hurdle_tcn_vs_baseline.csv": "dl_hurdle_tcn",
    "metrics_table_ml_hurdle_lgbm_vs_baseline.csv": "ml_hurdle_lgbm",
    "metrics_table_dl_gated_transformer_hurdle_vs_baseline.csv": "dl_gated_transformer_hurdle",
    "metrics_table_dl_hurdle_lstm_vs_baseline.csv": "dl_hurdle_lstm",
}

def _require_cols(df: pd.DataFrame, cols: list, fn: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {fn}: {missing}\nAvailable: {df.columns.tolist()}")

def _pick_first_row(df: pd.DataFrame, mask: pd.Series, kind: str, fn: str) -> pd.Series:
    sub = df[mask].copy()
    if sub.empty:
        raise ValueError(f"No {kind} row found in {fn}")
    return sub.iloc[0]

def _to_numeric_dict(row: pd.Series, cols: list) -> dict:
    out = {}
    for c in cols:
        out[c] = pd.to_numeric(row[c], errors="coerce")
    return out

rows = []

# 1) Add baseline row ONLY from BASELINE_SOURCE_FILE
p_base = METRICS_DIR / BASELINE_SOURCE_FILE
df_base = pd.read_csv(p_base)

_require_cols(df_base, ["Model"] + VALUE_COLS, BASELINE_SOURCE_FILE)

df_base["Model"] = df_base["Model"].astype(str).str.strip()
baseline_row = _pick_first_row(
    df_base,
    df_base["Model"].str.contains("Baseline", case=False, na=False),
    kind="baseline",
    fn=BASELINE_SOURCE_FILE,
)

rows.append(
    {
        "model_name": "baseline",
        "source_file": BASELINE_SOURCE_FILE,
        **_to_numeric_dict(baseline_row, VALUE_COLS),
    }
)

# 2) Add each model row from each file in IN_FILES (excluding baseline rows)
for fn in IN_FILES:
    p = METRICS_DIR / fn
    df = pd.read_csv(p)

    _require_cols(df, ["Model"] + VALUE_COLS, fn)

    df["Model"] = df["Model"].astype(str).str.strip()

    model_row = _pick_first_row(
        df,
        ~df["Model"].str.contains("Baseline", case=False, na=False),
        kind="model",
        fn=fn,
    )

    model_name = MODEL_NAME_BY_FILE.get(fn)
    if not model_name:
        # last resort: normalize the label from the CSV
        s = str(model_row["Model"]).strip().lower()
        s = s.replace(" (volume only)", "").replace(" ", "_")
        s = "".join(ch for ch in s if ch.isalnum() or ch == "_")
        model_name = s

    rows.append(
        {
            "model_name": model_name,
            "source_file": fn,
            **_to_numeric_dict(model_row, VALUE_COLS),
        }
    )

compact_df = pd.DataFrame(rows)

# 3) Save
OUT_CSV = METRICS_DIR / "consolidated_metrics_by_model.csv"
compact_df.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV.name)
display(compact_df)

Saved: consolidated_metrics_by_model.csv


,model_name,source_file,Active High,Active Low,Active Close,Active Volume,Active Total,Full High,Full Low,Full Close,Full Volume,Full Total
0,baseline,metrics_table_dl_gated_transformer_hurdle_vs_b...,0.227735,0.23061,0.231246,1.346832,0.509106,0.02761,0.02789,0.027855,0.089302,0.043164
1,dl_hurdle_tcn,metrics_table_dl_hurdle_tcn_vs_baseline.csv,0.227735,0.23061,0.231246,1.929596,0.654797,0.02761,0.02789,0.027855,0.092827,0.044045
2,ml_hurdle_lgbm,metrics_table_ml_hurdle_lgbm_vs_baseline.csv,0.227735,0.23061,0.231246,1.425700,0.528823,0.02761,0.02789,0.027855,0.074345,0.039425
3,dl_gated_transformer_hurdle,metrics_table_dl_gated_transformer_hurdle_vs_b...,0.227735,0.23061,0.231246,1.430663,0.530064,0.02761,0.02789,0.027855,0.083867,0.041805
4,dl_hurdle_lstm,metrics_table_dl_hurdle_lstm_vs_baseline.csv,0.227735,0.23061,0.231246,1.836802,0.631598,0.02761,0.02789,0.027855,0.091725,0.043770
